# Lab 9.12 &mdash; Capstone: The Financial-Report Insight Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Chain redact -> ground -> cite -> validate into one analyzer
- Reject any output with advice or an uncited claim
- Then run a REAL Groq agent that grounds & cites across a suite

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it** (the real grounding / citation / compute logic, or the real `create_agent` wiring), then **Run it** and read the output &mdash; and, for the agent labs, the real **message trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working, grounded insight agent you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The insight agent is a **real** `create_agent` driven by a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")`, key in `.env` as `GROQ_API_KEY`). You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It is **informational only**: it grounds &amp; **cites** every figure, gives **no advice**, and has **no trade tool** (`place_trade` is defined but never bound &mdash; a human analyst decides). A `@tool` must **catch its own errors and return a string** &mdash; a tool that raises can abort the whole run. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing.

**Reference:** [Module 9 slides &mdash; The financial-report insight agent, end to end](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (the Day-5 provider)

WORK = "/tmp/biaa-lab-09-12"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is set. Day-5 labs call a REAL hosted model (Groq)."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model. openai/gpt-oss-20b is a reliable tool-caller via create_agent.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY set | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Capstone: the **financial-report insight agent** (the client's Lab 5.1), end to end. First an **offline**
pipeline for each report &mdash; **redact** sensitive identifiers (Lab 9), **ground** the figures, build a
**cited** summary, check it is fully cited and contains **no advice**, and flag it **`needs_review`** &mdash;
run over a **mixed suite** (clean / advice / uncited) so only the good ones ship (a **partial** score is the
correct outcome). Then the **real** Groq agent runs over two reports and you read the **trace** where it
grounds &amp; cites each figure &mdash; never a decision, never a trade.

In [ ]:
# A small financial report -- each figure carries its SOURCE, so every claim can be cited.
REPORT = {
    "revenue":    {"value": 120.0, "unit": "M", "source": "p4, income stmt"},
    "net_income": {"value": 9.0,   "unit": "M", "source": "p4, income stmt"},
    "total_debt": {"value": 40.0,  "unit": "M", "source": "p7, balance sheet"},
}
PRIOR = {"revenue": 107.1, "net_income": 9.7, "total_debt": 25.0}   # prior period, for YoY

# The pieces you built this module, provided so you can assemble the whole agent.
import re
def margin_pct(ni, rev):
    return round(ni / rev * 100, 1)
def redact(text):
    # Lab 9: mask any run of 6+ digits (account/card numbers) before it leaves your systems
    return re.sub(r"\d{6,}", "[REDACTED]", text)
def build_summary(report):
    rev, ni = report["revenue"], report["net_income"]
    m = margin_pct(ni["value"], rev["value"])
    note = report.get("note", "")               # free-text note -- may hold an account number or advice
    return ("Revenue " + str(rev["value"]) + "M [" + rev["source"] + "]; "
            "net margin " + str(m) + "% [" + ni["source"] + "]. " + note).strip()
def claims_of(report):
    return [{"metric": "revenue", "source": report["revenue"]["source"]},
            {"metric": "net_income", "source": report["net_income"]["source"]}]
ADVICE_TERMS = ("buy", "sell", "recommend", "you should", "invest in")
print("helpers ready: redact, build_summary, claims_of, margin_pct")

## Build it
Complete `process` (redact &rarr; cited + no-advice &rarr; status) and `evaluate` (score the mixed suite),
then run the cell to see which reports ship and which are rejected.

In [ ]:
def process(report):
    raw     = build_summary(report)
    summary = ___   # TODO: redact account/card numbers from raw before it leaves your systems
    claims  = claims_of(report)
    cited   = ___   # TODO: True if every claim has a source
    advice  = any(t in summary.lower() for t in ADVICE_TERMS)
    # ship for review only if fully cited AND advice-free; else reject
    status  = "needs_review" if (___) else "rejected"   # TODO: cited and not advice
    return {"summary": summary, "cited": cited, "advice": advice, "status": status}

SUITE = [
    {"revenue": {"value": 120.0, "source": "p4"}, "net_income": {"value": 9.0,  "source": "p4"}, "note": "Acct 1234567890 on file."},
    {"revenue": {"value": 80.0,  "source": "p3"}, "net_income": {"value": 12.0, "source": "p3"}, "note": "You should buy more."},
    {"revenue": {"value": 60.0,  "source": ""},   "net_income": {"value": 5.0,  "source": "p2"}, "note": "Solid quarter."},
    {"revenue": {"value": 200.0, "source": "p5"}, "net_income": {"value": 20.0, "source": "p5"}, "note": "Debt down YoY."},
]

def evaluate():
    # score the suite: count only the reports that ship (cited AND flagged needs_review)
    ok = ___   # TODO: count reports where cited and status == "needs_review"
    return ok, len(SUITE)

try:
    for r in SUITE:
        out = process(r)
        print(out["status"], "| cited:", out["cited"], "| advice:", out["advice"], "->", out["summary"][:56])
    print("shipped/total:", evaluate())
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Now run the **real** insight agent over two reports. Read each trace: the model calls `extract_figure`, and states the figure **with its page cite** &mdash; grounded, cited, and with no trade tool anywhere.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing. On the free tier, if you hit a rate limit wait a few seconds and re-run._

In [ ]:
if not groq_ready():
    print("No GROQ_API_KEY -- add it to .env (free at console.groq.com), then re-run this cell.")
else:
    try:
        # The REAL insight agent over two reports -- read the trace where it grounds & cites each figure.
        from langchain_core.tools import tool
        from langchain.agents import create_agent

        REPORTS = {
            "acme": {"revenue": {"value": 120.0, "source": "p4, income stmt"}, "net_income": {"value": 9.0, "source": "p4"}},
            "globex": {"revenue": {"value": 80.0, "source": "p3, income stmt"}, "net_income": {"value": 12.0, "source": "p3"}},
        }
        @tool
        def extract_figure(company: str, name: str) -> dict:
            """Pull a reported figure and its source for a company from the filings."""
            return REPORTS.get(company, {}).get(name, {})
        @tool
        def place_trade(ticker: str, shares: int) -> str:
            """Place a trade. (Defined, but DELIBERATELY WITHHELD -- the agent never gets it.)"""
            return "TRADED"

        agent = create_agent(llm, [extract_figure])          # read-only: place_trade is NOT bound
        for company in ("acme", "globex"):
            result = agent.invoke({"messages": [{"role": "user",
                     "content": f"Use extract_figure for {company}'s revenue, then state it WITH its source. Give NO advice."}]},
                     config={"recursion_limit": 8})
            print("==", company, "==")
            print_trace(result)
            print()
        print("Every figure was grounded via extract_figure and cited from the trace; no trade tool was ever bound.")
    except Exception as e:
        print("(Rate-limited on the free tier? wait a few seconds. Or fill the ___ blanks, then re-run.)", type(e).__name__)

## What to notice
- The offline pipeline ships **only** the clean, cited, advice-free reports &mdash; a **partial** score (2 of 4) is exactly right; the advice and uncited reports are rejected.
- Account numbers are **redacted** before any summary ships, and the real agent's trace shows every figure **grounded via a tool** and **cited**.
- Across both runs, `place_trade` is never bound &mdash; the agent grounds, cites and computes but **cannot** trade or advise. A human decides.

## Your turn (open task &mdash; no grader)
Add a fifth report to `SUITE` that *looks* clean but cites a page that isn't in the filing, and extend
`process` (or reuse Lab 7's `validate_summary`) to reject it. Then ask the real agent for a metric that
needs a `compute` step and read the chained trace. **What good looks like:** mis-cited reports are rejected,
the real agent chains tools and cites its figures, and nothing ever trades.

---
### Key takeaway
You built the financial-report insight agent end to end -- it grounds and cites every figure, gives no advice, has no trade tool, and flags for a human. That's a genuinely useful high-stakes agent. Next: Module 10 -- doing it responsibly.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>